<a href="https://colab.research.google.com/github/OutForMilks/Hunger/blob/main/g2p.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setup**

**Google Collab**

In [6]:
# !git clone https://github.com/OutForMilks/Hunger.git
# %cd /content/Hunger

In [7]:
import pandas as pd
import os
from pathlib import Path
import glob

import torch
import torch.nn as nn
from torch.utils.data import WeightedRandomSampler
import sys

from src.prep_data import *
from src.decode import *
from src.train import *
from src.temperature import *
from src.evaluation import *

# **Prepare Data**

In [8]:
col_names = ["text", "phonetic", "language", "group"]

### Philippine Languages

In [9]:
ph_files = glob.glob("wikipron/filipino/*.tsv")
ph_languages = ["bikolano", "cebuano", "hiligaynon", "ilocano", "kapampangan", "pangasinan", "tagalog", "waray"]

temp_ph = []
for filename in ph_files:
    temp_df = pd.read_csv(filename, sep='\t', names=col_names, header=None)
    temp_df["language"] = Path(filename).stem
    temp_df["group"] = "philippine"
    temp_ph.append(temp_df)

ph_df = pd.concat(temp_ph, ignore_index=True)

print(ph_df.head())
print("\n")
print(ph_df["language"].unique())

      text       phonetic  language       group
0    Abril    ʔ a b ɾ i l  bikolano  philippine
1   Agosto  ʔ a ɡ o s t o  bikolano  philippine
2  Aguilar  ʔ a ɡ i l a ɾ  bikolano  philippine
3    Albay    ʔ a l b a j  bikolano  philippine
4   Alzaga  ʔ a l s a ɡ a  bikolano  philippine


<StringArray>
[   'bikolano',     'cebuano',  'hiligaynon',     'ilocano', 'kapampangan',
  'pangasinan',       'waray']
Length: 7, dtype: str


### Tagalog

In [10]:
input_dir_tgl = Path("wikipron/tagalog_cleaned.tsv")

df_tgl = pd.read_csv(input_dir_tgl, sep='\t', names=col_names, header=None)
df_tgl["language"] = "tagalog"
df_tgl["group"] = "philippine"

print(df_tgl.head())

   text phonetic language       group
0     g        ŋ  tagalog  philippine
1     m        m  tagalog  philippine
2    ng        ŋ  tagalog  philippine
3  'day    d a j  tagalog  philippine
4   'di    d i ʔ  tagalog  philippine


### Spanish

In [11]:
input_dir_spa = Path("wikipron/castillian_spanish_filtered.tsv")

df_spa = pd.read_csv(input_dir_spa, sep='\t', names=col_names, header=None)
df_spa["language"] = "spanish"
df_spa["group"] = "spanish"

print(df_spa.head())

     text   phonetic language    group
0  'tamo'    t a m o  spanish  spanish
1  'tamos  t a m o s  spanish  spanish
2    'tas      t a s  spanish  spanish
3    ACIP    a θ i p  spanish  spanish
4   ACNUR  a ɡ n u ɾ  spanish  spanish


### Splitting Tagalog

In [12]:
from sklearn.model_selection import train_test_split

# 70/15/15 split: peel off 15% for test, then 17.65% of the remaining 85% (~15% overall) for dev
train_dev_tgl, test_tgl = train_test_split(df_tgl, test_size=0.15, random_state=42)
train_tgl, dev_tgl = train_test_split(train_dev_tgl, test_size=0.1765, random_state=42)

ph_aux_df = ph_df.copy()
spa_aux_df = df_spa.copy()

# Each training mix = the tagalog train split + its auxiliary language group
tagalog_only_df = train_tgl.copy()
ph_df = pd.concat([train_tgl, ph_aux_df, spa_aux_df], ignore_index=True)

# Clean only now, so duplicates between the tagalog split and the aux data are caught too
for d in [tagalog_only_df, ph_df, df_spa]:
    d.drop_duplicates(subset=["text", "phonetic"], inplace=True)
    d.dropna(inplace=True)

In [13]:
input_dir = Path("data/input")
input_dir.mkdir(parents=True, exist_ok=True)

output_dir = Path("data/splits")
output_dir.mkdir(parents=True, exist_ok=True)

# Pool all training data, dedupe, then write one {lang}_train.tsv per language
# so convert_split can assemble any subset of languages into a training set.
combined_df = pd.concat(
    [ph_aux_df, spa_aux_df, train_tgl], ignore_index=True
).drop_duplicates(subset=["text", "phonetic"])

for lang, group_df in combined_df.groupby("language"):
    group_df[["text", "phonetic"]].to_csv(
        input_dir / f"{lang}_train.tsv", sep="\t", index=False, header=False
    )
    print(f"{lang}: {len(group_df)} pairs")

# The held-out tagalog dev/test splits -- every model is evaluated on these
dev_tgl[["text", "phonetic"]].to_csv(input_dir / "tagalog_dev.tsv", sep="\t", index=False, header=False)
test_tgl[["text", "phonetic"]].to_csv(input_dir / "tagalog_test.tsv", sep="\t", index=False, header=False)

bikolano: 5432 pairs
cebuano: 3031 pairs
hiligaynon: 165 pairs
ilocano: 827 pairs
kapampangan: 900 pairs
pangasinan: 158 pairs
spanish: 131869 pairs
tagalog: 13365 pairs
waray: 186 pairs


In [14]:
sets = {
    "ph_spanish": ph_languages + ["spanish"],
}

for set_name, langs in sets.items():
    out_dir = f"data/splits/{set_name}"
    print(f"Set: {set_name}")
    convert_split(input_dir, out_dir, langs, "train")

for split in ["dev", "test"]:
    out_dir = f"data/splits/tgl_{split}"
    print(f"Split: {split}")
    convert_split(input_dir, out_dir, ["tagalog"], split)

Set: ph_spanish
e:\College\Year 3\Term 3\DEEPLRN\Hunger
  bikolano train: 5432 pairs
  cebuano train: 3031 pairs
  hiligaynon train: 165 pairs
  ilocano train: 827 pairs
  kapampangan train: 900 pairs
  pangasinan train: 158 pairs
  tagalog train: 13365 pairs
  waray train: 186 pairs
  spanish train: 131869 pairs
  -> wrote 155933 lines to train.{src,tgt,lang}
Split: dev
e:\College\Year 3\Term 3\DEEPLRN\Hunger
  tagalog dev: 3259 pairs
  -> wrote 3259 lines to dev.{src,tgt,lang}
Split: test
e:\College\Year 3\Term 3\DEEPLRN\Hunger
  tagalog test: 3259 pairs
  -> wrote 3259 lines to test.{src,tgt,lang}


# **Train**

In [15]:
DATA = Path("data/splits")
OUTPUT = Path("models")
#seed is a var
STEPS = 200000
SAVE_STEP = 50000

BATCH_SIZE = 256
HIDDEN_SIZE = 512
FF_SIZE = 2048
N_LAYERS = 6
NUM_HEADS = 8

DROPOUT = 0.1
WARMUP = 4000
SMOOTHING = 0.1

# if len(xm.get_xla_supported_devices()) > 0:
#     DEVICE = "xla"
if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

xla = DEVICE == "xla"

print(DEVICE)

#loging is a var

CONFIG = {
    "data_path": str(DATA),
    "output_path": str(OUTPUT),

    "steps": STEPS,
    "save_step": SAVE_STEP,

    "batch_size": BATCH_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "ff_size": FF_SIZE,
    "n_layers": N_LAYERS,
    "num_heads": NUM_HEADS,

    "dropout": DROPOUT,
    "warmup": WARMUP,
    "label_smoothing": SMOOTHING,

    "device": DEVICE,
}

log_step = 1000

cuda


In [16]:
seeds = [42, 67, 888]
seed_checkpoint_results = {}

for seed in seeds:
    device = torch.device(DEVICE)
    torch.manual_seed(seed)

    # bf16 autocast on CUDA: tensor-core matmuls + roughly halved activation
    # memory; unlike fp16 it needs no GradScaler (same exponent range as fp32).
    use_amp = DEVICE == "cuda" and torch.cuda.is_bf16_supported()

    print(f"\nSeed: {seed}")
    DATA = Path(f"data/splits")
    OUTPUT = Path(f"models/seed_{seed}")

    os.makedirs(OUTPUT, exist_ok=True)
    
    expected_steps = list(range(SAVE_STEP, STEPS + 1, SAVE_STEP))
    expected_ckpts = [os.path.join(OUTPUT, f"ckpt_seed{seed}_checkpoint{s}.pt") for s in expected_steps]
    all_exist = all(os.path.exists(p) for p in expected_ckpts)


    if not all_exist:
        config_1 = dict(CONFIG)
        config_1["seed"] = seed
        config_1["data_path"] = str(DATA)
        config_1["output_path"] = str(OUTPUT)
        config_1["temperature"] = 3.0 

        # Re-seed per run so each model trains from the same init regardless of set order
        torch.manual_seed(seed)
        device = torch.device(DEVICE)

        # Vocab is built from this set's training data only (each set has its own
        # grapheme/phoneme inventory), then saved alongside the checkpoints.
        src_vocab, tgt_vocab = build_vocabs(
            os.path.join(DATA, "train.src"), os.path.join(DATA, "train.tgt"))
        save_vocabs(src_vocab, tgt_vocab, os.path.join(OUTPUT, "vocab.json"))

        train_ds = G2PDataset(os.path.join(DATA, "train.src"),
                                os.path.join(DATA, "train.tgt"),
                                src_vocab, tgt_vocab)
        # Fixed max lengths keep every batch the same shape -- required on XLA.
        # On GPU, pad each batch only to its own max instead: mean word length is
        # ~8 graphemes vs. a global max of ~48, so this skips most padding compute.
        max_src, max_tgt = compute_max_lengths(train_ds)
        print(f"src vocab={len(src_vocab)} tgt vocab={len(tgt_vocab)} "
                f"examples={len(train_ds)} | fixed shapes: src={max_src} tgt={max_tgt}")

        # Temperature sampling (T=5) flattens the language imbalance: low-resource
        # languages get sampled far above their natural frequency.
        weights = temperature_sampling_weights(
            os.path.join(DATA, "train.lang"), temperature=3.0
        )
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

        collate_fn = (make_fixed_collate(max_src, max_tgt) if xla
                    else make_dynamic_collate())
        loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                            collate_fn=collate_fn,
                            drop_last=True)

        model = Transformer(len(src_vocab), len(tgt_vocab), d_model=HIDDEN_SIZE,
                            n_heads=NUM_HEADS, d_ff=FF_SIZE,
                            n_layers=N_LAYERS, dropout=DROPOUT,
                            pad_id=PAD_ID).to(device)
        # lr=0 because NoamLR fully controls the learning rate (warmup then decay)
        opt = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
        sched = NoamLR(opt, HIDDEN_SIZE, WARMUP, xla=xla)
        criterion = LabelSmoothingLoss(len(tgt_vocab), PAD_ID, SMOOTHING).to(device)

        model.train()
        data_iter = infinite_loader(loader)
        for step in range(1, STEPS + 1):
            batch = next(data_iter)
            if xla:
                src, tgt_in, tgt_out, src_pad, tgt_pad = batch  # already on device
            else:
                src, tgt_in, tgt_out, src_pad, tgt_pad = (x.to(device) for x in batch)

            # Teacher forcing: tgt_in is the shifted target, tgt_out is what we score against
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_amp):
                logits = model(src, tgt_in, src_pad, tgt_pad)
                loss = criterion(logits, tgt_out)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            sched.step()  # calls xm.optimizer_step on XLA -> executes the graph

            if step % log_step == 0:
                # .item() forces a host sync; only do it at log intervals on TPU.
                print(f"step {step:>7}/{STEPS}  loss {loss.item():.4f}  "
                        f"lr {opt.param_groups[0]['lr']:.2e}")

            if step % SAVE_STEP == 0 or step == STEPS:
                # Checkpoint bundles weights + config + both vocabs, so evaluation
                # can rebuild the model from the .pt file alone.
                ckpt_path = os.path.join(OUTPUT, f"ckpt_seed{seed}_checkpoint{step}.pt")
                payload = {"model": model.state_dict(),
                            "config": config_1,
                            "src_vocab": src_vocab.to_dict(),
                            "tgt_vocab": tgt_vocab.to_dict()}
                torch.save(payload, ckpt_path)
                print(f"  saved {ckpt_path}")

        del model, opt, sched, criterion, loader, train_ds
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    else:
        print(f"\nSkipping seed {seed}, all checkpoints already exist")
    
    for step in expected_steps:
        ckpt_path = os.path.join(OUTPUT, f"ckpt_seed{seed}_checkpoint{step}.pt")
        per, wer, _, _ = batch_evaluate_checkpoint_beam(
            model_paths=[ckpt_path],
            src_path="./data/splits/tgl_dev/dev.src",
            tgt_path="./data/splits/tgl_dev/dev.tgt",
            device=device,
            batch_size=64,
        )
        seed_checkpoint_results[(seed, step)] = {"PER": per, "WER": wer}
        print(f"Seed={seed}, Checkpoint={step}: dev PER={per:.4f}  WER={wer:.4f}")


Seed: 42

Skipping seed 42, all checkpoints already exist
  decoded 64/3259
  decoded 128/3259
  decoded 192/3259
  decoded 256/3259
  decoded 320/3259
  decoded 384/3259
  decoded 448/3259
  decoded 512/3259
  decoded 576/3259
  decoded 640/3259
  decoded 704/3259
  decoded 768/3259
  decoded 832/3259
  decoded 896/3259
  decoded 960/3259
  decoded 1024/3259
  decoded 1088/3259
  decoded 1152/3259
  decoded 1216/3259
  decoded 1280/3259
  decoded 1344/3259
  decoded 1408/3259
  decoded 1472/3259
  decoded 1536/3259
  decoded 1600/3259
  decoded 1664/3259
  decoded 1728/3259
  decoded 1792/3259
  decoded 1856/3259
  decoded 1920/3259
  decoded 1984/3259
  decoded 2048/3259
  decoded 2112/3259
  decoded 2176/3259
  decoded 2240/3259
  decoded 2304/3259
  decoded 2368/3259
  decoded 2432/3259
  decoded 2496/3259
  decoded 2560/3259
  decoded 2624/3259
  decoded 2688/3259
  decoded 2752/3259
  decoded 2816/3259
  decoded 2880/3259
  decoded 2944/3259
  decoded 3008/3259
  decoded 3072/32

In [17]:
print("\nSeed per checkpoint summary:")
for (seed, ckpt), r in sorted(seed_checkpoint_results.items(), key=lambda x: x[1]["PER"]):
    print(f"Seed = {seed}, Checkpoint={ckpt}: PER={r['PER']:.4f}  WER={r['WER']:.4f}")

best_key = min(seed_checkpoint_results, key=lambda t: seed_checkpoint_results[t]["PER"])
best_seed, best_ckpt = best_key
print(f"\nBest combination: Seed={best_seed} Checkpoint={best_ckpt}")
print(f"\nPER={seed_checkpoint_results[best_key]['PER']:.4f}, WER={seed_checkpoint_results[best_key]['WER']:.4f})")


Seed per checkpoint summary:
Seed = 888, Checkpoint=200000: PER=0.0167  WER=0.1135
Seed = 42, Checkpoint=150000: PER=0.0167  WER=0.1117
Seed = 42, Checkpoint=100000: PER=0.0170  WER=0.1157
Seed = 888, Checkpoint=150000: PER=0.0174  WER=0.1169
Seed = 42, Checkpoint=50000: PER=0.0188  WER=0.1209
Seed = 888, Checkpoint=100000: PER=0.0192  WER=0.1154
Seed = 888, Checkpoint=50000: PER=0.0198  WER=0.1172
Seed = 67, Checkpoint=100000: PER=0.0203  WER=0.1105
Seed = 42, Checkpoint=200000: PER=0.0208  WER=0.1105
Seed = 67, Checkpoint=50000: PER=0.0229  WER=0.1212
Seed = 67, Checkpoint=200000: PER=0.0232  WER=0.1105
Seed = 67, Checkpoint=150000: PER=0.0237  WER=0.1141

Best combination: Seed=888 Checkpoint=200000

PER=0.0167, WER=0.1135)


# **Decode**

In [18]:
ensemble_paths = []
seeds = [42, 67, 888]

for seed in seeds:
    ensemble_paths.extend(sorted(glob.glob(f"models/seed_{seed}/ckpt_*.pt")))

print(f"\nEnsemble of {len(ensemble_paths)} checkpoints")
print(ensemble_paths)


Ensemble of 12 checkpoints
['models/seed_42\\ckpt_seed42_checkpoint100000.pt', 'models/seed_42\\ckpt_seed42_checkpoint150000.pt', 'models/seed_42\\ckpt_seed42_checkpoint200000.pt', 'models/seed_42\\ckpt_seed42_checkpoint50000.pt', 'models/seed_67\\ckpt_seed67_checkpoint100000.pt', 'models/seed_67\\ckpt_seed67_checkpoint150000.pt', 'models/seed_67\\ckpt_seed67_checkpoint200000.pt', 'models/seed_67\\ckpt_seed67_checkpoint50000.pt', 'models/seed_888\\ckpt_seed888_checkpoint100000.pt', 'models/seed_888\\ckpt_seed888_checkpoint150000.pt', 'models/seed_888\\ckpt_seed888_checkpoint200000.pt', 'models/seed_888\\ckpt_seed888_checkpoint50000.pt']


In [19]:
per, wer, refs, hyps = batch_evaluate_checkpoint_beam(
    model_paths=ensemble_paths,
    src_path="data/splits/tgl_test/test.src",
    tgt_path="data/splits/tgl_test/test.tgt",
    device=device,
    batch_size=64,
    beam=5,
)
print(f"\nEnsemble ({len(ensemble_paths)} models): PER={per:.4f}  WER={wer:.4f}")

  decoded 64/3259
  decoded 128/3259
  decoded 192/3259
  decoded 256/3259
  decoded 320/3259
  decoded 384/3259
  decoded 448/3259
  decoded 512/3259
  decoded 576/3259
  decoded 640/3259
  decoded 704/3259
  decoded 768/3259
  decoded 832/3259
  decoded 896/3259
  decoded 960/3259
  decoded 1024/3259
  decoded 1088/3259
  decoded 1152/3259
  decoded 1216/3259
  decoded 1280/3259
  decoded 1344/3259
  decoded 1408/3259
  decoded 1472/3259
  decoded 1536/3259
  decoded 1600/3259
  decoded 1664/3259
  decoded 1728/3259
  decoded 1792/3259
  decoded 1856/3259
  decoded 1920/3259
  decoded 1984/3259
  decoded 2048/3259
  decoded 2112/3259
  decoded 2176/3259
  decoded 2240/3259
  decoded 2304/3259
  decoded 2368/3259
  decoded 2432/3259
  decoded 2496/3259
  decoded 2560/3259
  decoded 2624/3259
  decoded 2688/3259
  decoded 2752/3259
  decoded 2816/3259
  decoded 2880/3259
  decoded 2944/3259
  decoded 3008/3259
  decoded 3072/3259
  decoded 3136/3259
  decoded 3200/3259
  decoded 3259/3

# **Evaluation**